In [2]:

import sqlite3
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Load data
conn = sqlite3.connect("/Users/Maazuddin/Downloads/inventory.db")

df = pd.read_sql_query(
    """
    SELECT Dollars, Quantity, Freight
    FROM vendor_invoice
    """,
    conn
)

conn.close()

# 2. Clean data
df = df.dropna(
    subset=["Dollars", "Quantity", "Freight"]
)

# Log1p requires Freight > -1.
# Freight should normally be nonnegative for this project.
df = df[df["Freight"] >= 0].copy()

# 3. Select features and target
X = df[["Dollars", "Quantity"]]
y = df["Freight"]

# Same split as previous experiments
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# 4. Train baseline Linear Regression
baseline = LinearRegression()
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

# 5. Train log-target Linear Regression
log_model = LinearRegression()

log_model.fit(
    X_train,
    np.log1p(y_train)
)

# Convert predictions back to freight amounts
log_predictions = np.expm1(
    log_model.predict(X_test)
)

# Freight cannot be negative
log_predictions = np.maximum(log_predictions, 0)

# 6. Evaluate both models on the original scale
def evaluate(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    r2 = r2_score(actual, predicted)

    print(f"\n{name}")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}")

evaluate(
    "Baseline Linear Regression",
    y_test,
    baseline_predictions
)

evaluate(
    "Log-Target Linear Regression",
    y_test,
    log_predictions
)

# 7. Compare the largest prediction errors
results = pd.DataFrame({
    "Actual_Freight": y_test,
    "Baseline_Prediction": baseline_predictions,
    "Log_Prediction": log_predictions
})

results["Baseline_Error"] = (
    results["Actual_Freight"]
    - results["Baseline_Prediction"]
).abs()

results["Log_Error"] = (
    results["Actual_Freight"]
    - results["Log_Prediction"]
).abs()

print("\nLargest baseline errors:")
print(
    results.nlargest(10, "Baseline_Error")
    .round(2)
    .to_string()
)




Baseline Linear Regression
MAE:  24.4592
RMSE: 124.4288
R²:   0.9700

Log-Target Linear Regression
MAE:  158933.5403
RMSE: 5025131.8889
R²:   -48897729.9456

Largest baseline errors:
      Actual_Freight  Baseline_Prediction  Log_Prediction  Baseline_Error   Log_Error
43           4613.45               938.93          122.10         3674.52     4491.35
61           1206.72               294.11           37.10          912.61     1169.62
4313         4581.05              5093.92       993823.68          512.87   989242.63
70            571.07               111.07           25.68          460.00      545.39
1820         4052.76              4505.79       295264.34          453.03   291211.58
401          4531.33              4112.02       164448.05          419.31   159916.72
2018         3473.39              3858.14        82761.09          384.75    79287.70
17            408.72                75.67           23.76          333.05      384.96
3428         5855.78              5527.25 